<a href="https://colab.research.google.com/github/Trangnguyen1402/AAI2025/blob/2026Fall/house_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

# Data source:
# Realtor.com housing data provided in realtor-data.csv
# File: realtor-data.csv

# Load the dataset
df = pd.read_csv("realtor-data.csv")

# Use sold homes in California
df = df[
    (df["status"] == "sold") &
    (df["state"] == "California")
].copy()

# Keep only rows with the required information
df = df[
    df["price"].notna() &
    df["house_size"].notna() &
    df["city"].notna()
]

# Remove invalid values
df = df[
    (df["price"] > 0) &
    (df["house_size"] > 0)
]

# Rename columns
df = df.rename(columns={
    "house_size": "square_footage",
    "city": "location"
})

# Remove extreme values and possible data errors
for column in ["price", "square_footage"]:
    lower_limit = df[column].quantile(0.01)
    upper_limit = df[column].quantile(0.99)

    df = df[
        df[column].between(lower_limit, upper_limit)
    ]

# Use 200 real records
df = df.sample(
    n=min(200, len(df)),
    random_state=42
)

print(f"Number of records used: {len(df)}")
print(f"Number of locations used: {df['location'].nunique()}")

# Features and target
X = df[["square_footage", "location"]]
y = df["price"]

# One-hot encode the location column
preprocessor = ColumnTransformer(
    transformers=[
        (
            "location",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["location"]
        )
    ],
    remainder="passthrough"
)

# Create the model pipeline
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Test the model
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"\nMean Absolute Error: ${mae:,.2f}")
print(f"R-squared Score: {r2:.3f}")

# Choose a location that exists in the dataset
new_location = df["location"].mode()[0]

# Predict the price of a new 2,000-square-foot house
new_house = pd.DataFrame({
    "square_footage": [2000],
    "location": [new_location]
})

predicted_price = model.predict(new_house)

print(
    f"\nPredicted price for a 2,000-square-foot house "
    f"in {new_location}: ${predicted_price[0]:,.2f}"
)

# Display model coefficients
location_features = (
    model.named_steps["preprocessor"]
    .named_transformers_["location"]
    .get_feature_names_out(["location"])
    .tolist()
)

feature_names = location_features + ["square_footage"]

coefficients = model.named_steps["regressor"].coef_

print("\nModel Coefficients:")

for feature, coefficient in zip(feature_names, coefficients):
    print(f"{feature}: {coefficient:,.2f}")

Number of records used: 200
Number of locations used: 132

Mean Absolute Error: $379,533.36
R-squared Score: 0.130

Predicted price for a 2,000-square-foot house in Los Angeles: $1,687,601.76

Model Coefficients:
location_Alameda: -24,221.54
location_Alhambra: 230,113.12
location_Anaheim: 111,888.99
location_Antelope: -433,653.87
location_Antioch: -449,775.04
location_Aptos: 471,588.16
location_Bakersfield: -173,765.37
location_Ben Lomond: 192,792.39
location_Canoga Park: -345,874.32
location_Cathedral City: -412,610.18
location_Chino Hills: -208,393.75
location_Chula Vista: -313,576.69
location_Compton: -42,855.45
location_Concord: -135,604.55
location_Corona: -278,569.36
location_Costa Mesa: 534,077.68
location_Cressey: -725,358.64
location_Daly City: 584,105.18
location_Dana Point: 948,923.43
location_Danville: 596,710.91
location_Davis: -4,789.89
location_Eastvale: -529,805.84
location_El Cajon: -395,178.53
location_El Dorado Hills: -334,616.48
location_Elk Grove: -176,408.89
locat